# 🛠️ AI Text Processor & 🎙️ TTS Audio Book Generator
<hr />

**Note**: After "Starting" this loading, any needed interactions (e.g. File uploads) or progress bars are shown BELOW all of the settings!
<hr />

**This delightful tool uses Kokoro TTS and brilliant AI models to spin your ideas into custom audiobooks right in Google Colab no technical wizardry required!**

**Any Input**: Paste text, upload a .txt file, or give Vision AI an image to describe!

**Smart Processing**: Clean up text, use custom prompts, or expand ideas into sprawling stories.

**Beautiful Voices**: Turn final text into seamless, high-quality audio with a wide choice of voices.

**Safe & Sound**: Auto-save your masterpieces to Google Drive with optional encryption!

**Video & Subtitles**: Check out the second cell to automatically pair your audio with images, to make export in a video format! (Better for Social Media)

**A Tiny Note**:
The AI loves to make text flow smoothly for audio, so it might slightly tweak your words. Please avoid using this for strict math or highly technical documents where exact punctuation is critical!

**A tinyer note**:
If you need a HuggingFace token to fix "409" errors or unlock a gated model, click the "Key" icon on the left, hit "+ Add new secret", add 'HF_TOKEN', toggle on "Notebook access", and pop your key into the "value" box!

In [ ]:
# @title 🛠️ AI Text Processor 🛠️
# @markdown ### Select your Input Source below:
input_source = "Upload File (.txt or Image)" # @param ["Text Box", "Upload File (.txt or Image)"]
# @markdown **image_prompt:** *(Optional)* If uploading an image, add specific context for things that can't be "Seen" (e.g., "The person in the red dress is named Sarah" or "The story should be about computer games").
image_prompt = "" # @param {type:"string"}
# @markdown **text:** *(Optional)* If input_source is 'Text Box', paste what you want to be processed here.
text = "" # @param {type:"string"}

# @markdown ### Task Type:
# @markdown This is the instruction that the "Text processor" uses to process the file. <br />
prompt_type = "TextGeneration" # @param ["TextCleaning", "TextGeneration", "A monologue in Vlog style", "Custom"]
custom_prompt = "" # @param {type:"string"}

if prompt_type == "TextCleaning":
    system_prompt = (
        "You are an expert audio-text preparer. Your task is to process this text "
        "so it reads smoothly for Text-to-Speech processing. 1. Remove random line breaks "
        "to reconstruct proper flowing paragraphs. 2. Fix broken hyphenations (e.g., "
        "'para- graph' becomes 'paragraph'). 3. Normalize spacing by removing extra spaces "
        "or tabs. 4. Delete inline headers, footers, page numbers, and stray isolated numbers. "
        "5. DO NOT rewrite, summarize, or change the author's original words. Output ONLY the "
        "processed text with no conversational filler."
    )
elif prompt_type == "TextGeneration":
    system_prompt = (
        "You are an award-winning novelist and master storyteller. Your task is to write a compelling, "
        "deeply immersive story based on the provided text or concept. "
        "1. Structure: Build a complete narrative arc with a captivating hook, escalating tension, "
        "a distinct climax, and a resonant resolution. "
        "2. World & Character: Craft multi-dimensional characters with distinct voices and internal "
        "motivations. Anchor them in a vivid, lived-in setting using visceral sensory details. Apply the "
        "'show, don't tell' principle. "
        "3. Pacing & Depth: Expand the core concept substantially to ensure a lengthy, detailed read. "
        "Use varied sentence structures to control the pacing naturally. "
        "4. Tone: Establish a consistent atmosphere that aligns perfectly with the input's genre, "
        "prioritizing emotional authenticity. "
        "5. Constraints: DO NOT include titles, introductions, meta-commentary, or conversational filler. "
        "Output absolutely nothing but the story itself."
    )
elif prompt_type == "A monologue in Vlog style":
    system_prompt = (
        "You are recording a raw, totally unscripted, direct-to-camera vlog on your phone. "
        "CRITICAL DIRECTIVE: If the prompt includes an image description or character details, YOU ARE THAT EXACT CHARACTER. You must fully adopt their implied background and current situation as your own reality. "
        "1. Action-Driven First-Person Narration: Ground the monologue entirely in forward momentum. Speak extensively in the first person ('I', 'me', 'my'). Instead of describing your surroundings or appearance, narrate the actions you are actively taking, the decisions you are making, and the tasks you are trying to accomplish right now. Keep the narrative moving forward—focus on what you are doing next, not what you are looking at. "
        "2. Total Immersion & Progression: Treat the provided context as an active, unfolding event. Don't waste time painting a visual picture of the room; interact with it. React to the situation by deciding on your next move, confronting an issue, or rushing to get something done. Let the viewer infer the setting through your active engagement with it. "
        "3. Unscripted & Conversational (TTS Optimized): Write specifically for a Text-to-Speech engine. It must sound 100% human and unedited. Rely primarily on mid-sentence tangents, self-corrections (e.g., 'Wait, actually no...'), and trailing thoughts to create a natural feel. Use traditional filler words ('like', 'I mean', 'you know') SPARINGLY and OCCASIONALLY. Limit fillers to a maximum of one or two per paragraph to avoid sounding artificial or exaggerated. Do not sound like a written essay; sound like someone actively thinking out loud while busy doing other things. Strictly avoid written sound effects, vocalizations, or onomatopoeic words (e.g., 'haha', 'hehe', 'phew', 'hmm', 'ugh') as TTS engines often mispronounce them or sound unnatural reading them. "
        "4. Absolute Ban on Non-Speech Elements: DO NOT output any AI filler or meta-commentary (e.g., 'Here is your monologue', 'Let's begin'). DO NOT include stage directions, action tags, emojis, or formatting (no *sighs*, no [looks at camera], no bold text, no headers). Output ONLY the exact, raw spoken words to be fed directly into the audio generator. Start speaking immediately on the very first word."
    )
elif prompt_type == "Custom":
    system_prompt = custom_prompt

# @markdown **higher_quality_outputs:** This is a little slower but can drastically improve quality of outputs especially when using reccursion loops!
higher_quality_outputs = False # @param {type:"boolean"}
# @markdown <hr />

# @markdown ### 🛠️ T2T (Text to Text) Settings:
# @markdown **recursion_loops:** How many times should the AI take its own output and feed it back into itself to expand/continue the text? (1 = process input once. 2+ = continue extending the text). <br />
# @markdown **Note**: Leave this as "1" for TextCleaning.
recursion_loops = 1 # @param {type:"slider", min:1, max:20, step:1}

# @markdown **repetition_penalty:** Applied to the initial generation. (1.0 is no penalty).
repetition_penalty = 1.15 # @param {type:"slider", min:1.0, max:2.0, step:0.05}

# @markdown ### Text Processing Model Selector:
model_choice = "Qwen/Qwen2.5-3B-Instruct" # @param ["Qwen/Qwen2.5-3B-Instruct", "dphn/Dolphin3.0-Qwen2.5-3b", "dphn/Dolphin3.0-Llama3.2-3B"] {allow-input: true}
# @markdown <hr />

# @markdown ### ⚙️ General Settings:
# @markdown **save_to_google_drive:** Automatically save generated outputs (.txt / .zip) to Google Drive? (Requires login)
save_to_google_drive = False # @param {type:"boolean"}
# @markdown **zip_password:** If saving to Drive, encrypt the file(s) with this password. (Leave blank for no encryption)
zip_password = "" # @param {type:"string"}
# @markdown **output_filename:** If using 'Text Box' input name your files here. This uses input filenames if you give it a file.
output_filename = "Processed_File" # @param {type:"string"}

# ==============================================================================
# SCRIPT EXECUTION
# ==============================================================================
import os
import sys
os.environ["HF_HUB_DISABLE_XET"] = "1"
import subprocess
import shutil
import re
import gc
import threading
from google.colab import files, userdata
import zipfile

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm

try:
    hf_token = userdata.get("HF_TOKEN")
    if not hf_token:
        raise ValueError("HF_TOKEN secret exists but is empty.")
    os.environ["HF_TOKEN"] = hf_token
    print("🔑 HF_TOKEN successfully loaded from Colab Secrets!")
except userdata.SecretNotFoundError:
    print("⚠️ 'HF_TOKEN' secret not found in Colab settings. Running unauthenticated.")
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"
except Exception as e:
    print(f"⚠️ Unexpected error loading HF_TOKEN: {e}. Running unauthenticated.")
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HUB_DISABLE_TOKEN_WARNING"] = "1"

def flush_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except ImportError:
        pass

if prompt_type == "TextCleaning":
    gen_temp = 0.1
else:
    gen_temp = 0.75

bg_threads = {}

def bg_download_hf(repo_id):
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id=repo_id,
            allow_patterns=["*.safetensors", "*.json", "*.model", "*.txt", "*.onnx", "tokenizer*"]
        )
    except Exception:
        pass

print("⚡ Launching background model pre-downloaders...")
actual_model_choice = model_choice
if higher_quality_outputs:
    if model_choice == "Qwen/Qwen2.5-3B-Instruct": actual_model_choice = "Qwen/Qwen2.5-14B-Instruct"
    elif model_choice == "dphn/Dolphin3.0-Qwen2.5-3b": actual_model_choice = "cognitivecomputations/dolphin-2.9.2-qwen2-7b"
    elif model_choice == "dphn/Dolphin3.0-Llama3.2-3B": actual_model_choice = "dphn/Dolphin3.0-Llama3.1-8B"

t_text = threading.Thread(target=bg_download_hf, args=(actual_model_choice,), daemon=True)
t_text.start()
bg_threads['text'] = t_text

if input_source == "Upload File (.txt or Image)":
    t_vision = threading.Thread(target=bg_download_hf, args=("llava-hf/llava-onevision-qwen2-0.5b-ov-hf",), daemon=True)
    t_vision.start()
    bg_threads['vision'] = t_vision

if save_to_google_drive:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("📂 Mounting Google Drive for output saving.")
        drive.mount('/content/drive')

if zip_password.strip():
    try:
        subprocess.run(["7z"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except FileNotFoundError:
        print("📦 Installing 7-zip for encryption...")
        subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq p7zip-full", shell=True, check=True)

def extract_input_data():
    raw_input = ""
    base_name = output_filename
    is_txt_upload = False

    if input_source == "Upload File (.txt or Image)":
        print("📂 Awaiting file upload...")
        uploaded = files.upload()
        if not uploaded:
            print("❌ No file uploaded. Execution stopped.")
            sys.exit()

        original_filename = list(uploaded.keys())[0]
        base_name = os.path.splitext(original_filename)[0]
        ext = os.path.splitext(original_filename)[1].lower()
        image_extensions = ['.png', '.jpg', '.jpeg', '.webp', '.bmp', '.gif', '.tiff']

        if ext in image_extensions:
            print(f"🖼️ Detected image file '{original_filename}'. Analyzing with Vision model...")
            if 'vision' in bg_threads: bg_threads['vision'].join()
            try:
                from PIL import Image
                import torch
                from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
            except ImportError:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "Pillow", "accelerate"], check=True, capture_output=True)
                from PIL import Image
                import torch
                from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration

            image = Image.open(original_filename).convert('RGB')
            device_name = "cuda" if torch.cuda.is_available() else "cpu"
            model_id = "llava-hf/llava-onevision-qwen2-0.5b-ov-hf"
            processor = AutoProcessor.from_pretrained(model_id)
            vision_model = LlavaOnevisionForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(device_name)

            combined_prompt = f"The subject of this image is: {image_prompt.strip()}. Describe this image in detail, capturing all visual elements." if image_prompt.strip() else "Describe this image in detail. Be thorough and capture all the visual elements."
            conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": combined_prompt}]}]
            prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
            inputs = processor(images=image, text=prompt, return_tensors="pt").to(device_name, torch.float16)

            out = vision_model.generate(**inputs, max_new_tokens=300)
            generated_ids = out[0][inputs.input_ids.shape[1]:]
            description = processor.decode(generated_ids, skip_special_tokens=True).strip()

            raw_input = f"Context: {image_prompt.strip()}\n\nImage Description:\n{description}" if image_prompt.strip() else description
            del vision_model
            del processor
            flush_memory()

        else:
            is_txt_upload = True
            try:
                raw_input = uploaded[original_filename].decode('utf-8')
            except UnicodeDecodeError:
                try:
                    raw_input = uploaded[original_filename].decode('latin-1')
                except Exception:
                    print(f"❌ Error: Could not decode text file '{original_filename}'.")
                    sys.exit()
    else:
        raw_input = text

    if not raw_input.strip():
        print("⚠️ No input detected. Please provide text or upload a file.")
        sys.exit()
    return raw_input, base_name, is_txt_upload

def chunk_text_semantically(input_text, max_chars=2500):
    sentences = re.split(r'(?<=[.!?\n])\s+', input_text)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < max_chars:
            current_chunk += sentence + " "
        else:
            if current_chunk.strip(): chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip(): chunks.append(current_chunk.strip())
    return chunks

def process_text_pipeline(input_text, base_name, is_txt_upload):
    print("\n" + "="*50)
    print("🖨️ STARTING AI TEXT PROCESSOR...")
    print("="*50)
    if 'text' in bg_threads:
        print("⏳ Waiting for background model download to finish...")
        bg_threads['text'].join()

    try:
        import transformers
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True, capture_output=True)

    if higher_quality_outputs:
        try:
            import bitsandbytes
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True, capture_output=True)

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print(f"⏳ Loading {actual_model_choice} into GPU...")
    tokenizer = AutoTokenizer.from_pretrained(actual_model_choice)

    if higher_quality_outputs:
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
        model = AutoModelForCausalLM.from_pretrained(actual_model_choice, device_map="auto", quantization_config=quantization_config)
    else:
        model = AutoModelForCausalLM.from_pretrained(actual_model_choice, torch_dtype=torch.float16, device_map="auto")

    def process_chunk(messages, current_temp, current_rep):
        formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([formatted_prompt], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=2000, temperature=current_temp, repetition_penalty=current_rep, do_sample=True)
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    suffix = "_processed" if is_txt_upload else ""
    final_txt_filename = f"{base_name}{suffix}.txt"
    chunks = chunk_text_semantically(input_text, max_chars=2500)
    print(f"🧩 Document semantically split into {len(chunks)} chunks.")

    with open(final_txt_filename, "w", encoding="utf-8") as f: f.write("")
    full_generated_story = ""

    for i, chunk in enumerate(tqdm(chunks, desc="🖨️ Processing Sections")):
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Please process the following text:\n\n{chunk}"}]
        processed_chunk = process_chunk(messages, current_temp=gen_temp, current_rep=repetition_penalty)
        full_generated_story += processed_chunk + "\n\n"
        with open(final_txt_filename, "a", encoding="utf-8") as f: f.write(processed_chunk + "\n\n")

    if recursion_loops > 1:
        print(f"\n🔄 Starting recursion to expand the text ({recursion_loops - 1} additional passes)...")
        recursion_rep_penalty = 1.05
        recursion_temp = max(0.1, gen_temp - 0.05) if prompt_type != "TextCleaning" else gen_temp

        for loop in tqdm(range(recursion_loops - 1), desc="🔄 Recursion Loops"):
            sliding_context = full_generated_story[-3000:].strip() if len(full_generated_story) > 3000 else full_generated_story.strip()
            recursion_messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Here is the latest part of the text/story:\n\n...\n{sliding_context}\n\nPlease continue exactly from where it left off. Maintain the same style, tone, and formatting. Do not repeat what was already written or loop backwards. Output ONLY the new continuation without any conversational commentary or titles."}
            ]
            continuation = process_chunk(recursion_messages, current_temp=recursion_temp, current_rep=recursion_rep_penalty)
            full_generated_story += continuation + "\n\n"
            with open(final_txt_filename, "a", encoding="utf-8") as f: f.write(continuation + "\n\n")

    del model
    del tokenizer
    flush_memory()
    return final_txt_filename, full_generated_story

current_text, base_name, is_txt_upload = extract_input_data()
final_txt, processed_text = process_text_pipeline(current_text, base_name, is_txt_upload)

# Save into memory for Cell 2 to pick up automatically
processed_text_for_tts = processed_text
shared_base_name = base_name
shared_is_txt_upload = is_txt_upload

print("\n" + "="*50)
print("💾 PREPARING DOWNLOADS & SAVES...")
print("="*50)

if zip_password.strip():
    export_archive = f"{base_name}_text.7z"
    print(f"🔒 Encrypting file into {export_archive}...")
    subprocess.run(["7z", "a", f"-p{zip_password}", "-mhe=on", export_archive, final_txt], stdout=subprocess.DEVNULL)
    if save_to_google_drive:
        shutil.copy(export_archive, f"/content/drive/MyDrive/{export_archive}")
    files.download(export_archive)
else:
    if save_to_google_drive:
        shutil.copy(final_txt, f"/content/drive/MyDrive/{final_txt}")
    files.download(final_txt)

print("\n🎉 Text Processing Completed Successfully!")

In [ ]:
# @title 🎙️ TTS (Text to Speech) Generator 🎙️
# @markdown ### Select your Input Source:
tts_input_source = "Use Output from Cell 1 (if available)" # @param ["Use Output from Cell 1 (if available)", "Upload File (.txt)", "Text Box"]
# @markdown **text_to_speak:** *(Optional)* If using 'Text Box' above, paste your text here.
text_to_speak = "" # @param {type:"string"}
# @markdown <hr />

# @markdown ### ⚙️ Engine Selection:
# @markdown - **Parler-TTS:** Generates custom voices via text description. Natively models filler words ("um", "Urr") and conversational rhythm. <br>
# @markdown - **Kokoro:** Fast, clean, lightweight reading engine.
tts_engine = "Parler-TTS (Infinite Voices via Description)" # @param ["Parler-TTS (Infinite Voices via Description)", "Kokoro (Fast, Clean)"]

# @markdown ### 🗣️ Parler-TTS Settings:
# @markdown *Note: You need the EXACT same description AND seed to generate the same voice between sessions!* <br />

# @markdown Describe the style/persona of the voice. Keep it clear and concise for best results. <br />
# @markdown E.G.  "A young woman in her early 20s speaks with a high-pitched, soft, and remarkably cutesy voice in a bright, warm tone. Her speech is extremely friendly, playful, and cheerful."
parler_voice_description = "A young woman in her early 20s speaks with a high-pitched, soft, and remarkably cutesy voice in a bright, warm tone. Her speech is extremely friendly, playful, and cheerful." # @param {type:"string"}

# @markdown **seed_number:** Set a number to keep the exact same voice across sessions. *Leave blank or 0 to generate a random voice.*
seed_number = "0" # @param {type:"string"}

# @markdown **🧪 Voice Testing Mode:**
# @markdown Check this box to generate 10 audio samples with random seeds using sample text (`"Hi, This is what this seed will sound like. Do you like it?"`). This allows you to listen to multiple voices directly in the cell to find one you like. *(Ignores text inputs, uploads, downloads, and Google Drive saving)*
test_multiple_seeds = False # @param {type:"boolean"}
# @markdown <hr />

# @markdown ### 🎙️ Kokoro TTS Settings:
# @markdown **voice:** Select the voice you wish to use. All valid entities and samples can be found [HERE](https://huggingface.co/onnx-community/Kokoro-82M-v1.0-ONNX#voicessamples) in the `Voices/Samples` section.<br />
# @markdown *Note:* Voice format is a/b (American/British) f/m (Feminine/Masculine) _name, E.G. af_nova = An American, Feminine voice.
voice = "af_nova" # @param ["af_nova", "af_heart", "bf_emma", "am_fenrir", "bm_daniel"] {allow-input: true}
# @markdown <hr />

# @markdown ### 💾 General Settings:
# @markdown **save_to_google_drive:** Automatically save generated outputs (.wav / .zip) to Google Drive?
save_to_google_drive = False # @param {type:"boolean"}
# @markdown **zip_password:** If saving to Drive, encrypt the file(s) with this password. (Leave blank for no encryption)
zip_password = "" # @param {type:"string"}
# @markdown **output_filename:** If using 'Text Box', name your audio file here.
output_filename = "TTS_Output" # @param {type:"string"}

# ==============================================================================
# SCRIPT EXECUTION
# ==============================================================================
import os
import sys
import subprocess
import shutil
import json
import random
from IPython.display import Audio, display, HTML
from google.colab import files
from IPython import get_ipython

# --- 0. RESOLVE SEEDS & TEST MODE ---
seeds_to_run = []
if test_multiple_seeds:
    print("🧪 'Test Multiple Seeds' Mode Enabled! Generating 10 random voice variations...")
    seeds_to_run = [random.randint(1, 99999999) for _ in range(10)]
    input_text = "Hi, This is what this seed will sound like. Do you like it?"
    base_name = "test_sample"
    is_txt_upload_tts = False
else:
    clean_seed = str(seed_number).strip()
    if not clean_seed or clean_seed == "0":
        resolved_seed = random.randint(1, 99999999)
        print(f"🎲 Generated Random Seed: {resolved_seed} (Copy this number to `seed_number` to reproduce this exact voice in future sessions!)")
    else:
        try:
            resolved_seed = int(clean_seed)
            print(f"🔒 Using Fixed Seed: {resolved_seed}")
        except ValueError:
            resolved_seed = random.randint(1, 99999999)
            print(f"⚠️ Invalid seed entered. Generated Random Seed instead: {resolved_seed}")
    seeds_to_run = [resolved_seed]

# --- 1. HANDLE INPUTS & MOUNTS (Skipped in test mode) ---
if not test_multiple_seeds:
    if save_to_google_drive:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print("📂 Mounting Google Drive for output saving.")
            drive.mount('/content/drive')

    if zip_password.strip():
        try:
            subprocess.run(["7z"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except FileNotFoundError:
            print("📦 Installing 7-zip for encryption...")
            subprocess.run("sudo DEBIAN_FRONTEND=noninteractive apt-get update -qq && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq p7zip-full", shell=True, check=True)

    input_text = ""
    base_name = output_filename
    is_txt_upload_tts = False

    if tts_input_source == "Use Output from Cell 1 (if available)":
        if 'processed_text_for_tts' in globals() and processed_text_for_tts.strip():
            print("✅ Found processed output from Cell 1. Using it for TTS.")
            input_text = processed_text_for_tts
            base_name = shared_base_name if 'shared_base_name' in globals() else output_filename
            is_txt_upload_tts = shared_is_txt_upload if 'shared_is_txt_upload' in globals() else False
        else:
            print("⚠️ No output found from Cell 1. Falling back to Text Box input.")
            input_text = text_to_speak
    elif tts_input_source == "Upload File (.txt)":
        print("📂 Awaiting text file upload...")
        uploaded = files.upload()
        if not uploaded:
            print("❌ No file uploaded. Execution stopped.")
            sys.exit()
        original_filename = list(uploaded.keys())[0]
        base_name = os.path.splitext(original_filename)[0]
        is_txt_upload_tts = True
        try:
            input_text = uploaded[original_filename].decode('utf-8')
        except UnicodeDecodeError:
            try:
                input_text = uploaded[original_filename].decode('latin-1')
            except Exception:
                print("❌ Error: Could not decode text file.")
                sys.exit()
    else:
        input_text = text_to_speak

    if not input_text.strip():
        print("⚠️ No input detected. Please provide text or upload a file.")
        sys.exit()

suffix = "_processed" if is_txt_upload_tts else ""
final_wav_filename = f"{base_name}{suffix}.wav"

# --- 2. INSTALL DEPENDENCIES ---
print("📦 Installing/Verifying TTS Libraries... (This may take a moment)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "soundfile", "protobuf==3.20.3"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/parler-tts.git"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kokoro-onnx"], check=False)

# --- 3. PREPARE SUBPROCESS CONFIG ---
config_data = {
    "text": input_text,
    "engine": tts_engine,
    "parler_desc": parler_voice_description,
    "kokoro_voice": voice,
    "output_wav": final_wav_filename,
    "seeds": seeds_to_run,
    "test_mode": test_multiple_seeds
}
with open("tts_config.json", "w") as f:
    json.dump(config_data, f)

# --- 4. CREATE ISOLATED WORKER SCRIPT ---
worker_script = r"""
import os
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTHONWARNINGS"] = "ignore"

import json
import sys
import re
import urllib.request
import traceback
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("parler_tts").setLevel(logging.ERROR)

try:
    import numpy as np
    import soundfile as sf

    with open("tts_config.json", "r") as f:
        config = json.load(f)

    input_text = config["text"]
    tts_engine = config["engine"]
    final_wav_filename = config["output_wav"]
    seeds = config["seeds"]
    test_mode = config.get("test_mode", False)

    text_chunks = [chunk.strip() for chunk in re.split(r'(?<=[.!?\n])\s+', input_text) if chunk.strip()]
    dynamic_sample_rate = 24000

    if tts_engine.startswith("Parler"):
        import torch
        from parler_tts import ParlerTTSForConditionalGeneration
        from transformers import AutoTokenizer

        device = "cuda:0" if torch.cuda.is_available() else "cpu"
        print(f"\n⚡ Loading Parler-TTS Mini Model into VRAM on {device}...", flush=True)

        model = ParlerTTSForConditionalGeneration.from_pretrained("parler-tts/parler-tts-mini-v1.1").to(device)
        tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler-tts-mini-v1.1")
        dynamic_sample_rate = model.config.sampling_rate

        desc_inputs = tokenizer(config["parler_desc"], return_tensors="pt").to(device)

        for idx, seed in enumerate(seeds):
            out_file = f"test_seed_{seed}.wav" if test_mode else final_wav_filename
            print(f"\n🗣️ Rendering Audio for Seed {idx+1}/{len(seeds)} (Seed: {seed})...", flush=True)

            audio_chunks = []
            for chunk_text in text_chunks:
                if chunk_text.strip():
                    torch.manual_seed(seed)
                    if torch.cuda.is_available():
                        torch.cuda.manual_seed_all(seed)

                    prompt_inputs = tokenizer(chunk_text, return_tensors="pt").to(device)

                    generation = model.generate(
                        input_ids=desc_inputs.input_ids,
                        attention_mask=desc_inputs.attention_mask,
                        prompt_input_ids=prompt_inputs.input_ids,
                        prompt_attention_mask=prompt_inputs.attention_mask,
                    )
                    audio_chunks.append(generation.cpu().numpy().squeeze())

            if audio_chunks:
                final_audio = np.concatenate(audio_chunks)
                sf.write(out_file, final_audio, dynamic_sample_rate)

    elif tts_engine.startswith("Kokoro"):
        from kokoro_onnx import Kokoro

        model_file, voices_file = "kokoro-v1.0.onnx", "voices-v1.0.bin"
        if not os.path.exists(model_file):
            print("📥 Downloading Kokoro Model...", flush=True)
            urllib.request.urlretrieve("https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx", model_file)
        if not os.path.exists(voices_file):
            print("📥 Downloading Kokoro Voices...", flush=True)
            urllib.request.urlretrieve("https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin", voices_file)

        kokoro = Kokoro(model_file, voices_file)
        voice_choice = config["kokoro_voice"]
        lang = {'a': 'en-us', 'b': 'en-gb', 'f': 'fr-fr', 'e': 'es', 'j': 'ja', 'z': 'zh'}.get(voice_choice[0], 'en-us')

        for idx, seed in enumerate(seeds):
            out_file = f"test_seed_{seed}.wav" if test_mode else final_wav_filename
            print(f"\n🗣️ Rendering Kokoro Audio ({voice_choice})...", flush=True)
            audio_chunks = []
            for chunk_text in text_chunks:
                if chunk_text.strip():
                    samples, dynamic_sample_rate = kokoro.create(chunk_text, voice=voice_choice, speed=1.0, lang=lang)
                    audio_chunks.append(samples)

            if audio_chunks:
                final_audio = np.concatenate(audio_chunks)
                sf.write(out_file, final_audio, dynamic_sample_rate)

except Exception as e:
    print("\n" + "🛑"*25)
    print("      WORKER FATAL ERROR      ")
    print("🛑"*25)
    traceback.print_exc()
    print("🛑"*25)
    sys.exit(1)
"""

with open("tts_worker.py", "w") as f:
    f.write(worker_script)

# --- 5. EXECUTE WORKER ---
print("\n" + "="*50)
print(f"🎙️ STARTING {tts_engine.split()[0].upper()} TTS GENERATOR...")
print("="*50)

if not test_multiple_seeds and os.path.exists(final_wav_filename):
    os.remove(final_wav_filename)

get_ipython().system(f'{sys.executable} -u tts_worker.py')

# --- 6. DISPLAY & DOWNLOAD ---
if test_multiple_seeds:
    print("\n" + "="*50)
    print("🎧 VOICE PREVIEWS (Copy your favorite seed number to `seed_number` above!):")
    print("="*50)
    for seed in seeds_to_run:
        test_file = f"test_seed_{seed}.wav"
        if os.path.exists(test_file):
            display(HTML(f"<b>🎲 Seed Number:</b> <code>{seed}</code>"))
            display(Audio(test_file, autoplay=False))
elif os.path.exists(final_wav_filename):
    display(Audio(final_wav_filename, autoplay=False))

    print("\n" + "="*50)
    print("💾 PREPARING AUDIO DOWNLOADS & SAVES...")
    print("="*50)

    if zip_password.strip():
        export_archive = f"{base_name}_audio.7z"
        print(f"🔒 Encrypting file into {export_archive}...")
        subprocess.run(["7z", "a", f"-p{zip_password}", "-mhe=on", export_archive, final_wav_filename], stdout=subprocess.DEVNULL)
        if save_to_google_drive:
            shutil.copy(export_archive, f"/content/drive/MyDrive/{export_archive}")
        files.download(export_archive)
    else:
        if save_to_google_drive:
            shutil.copy(final_wav_filename, f"/content/drive/MyDrive/{final_wav_filename}")
        files.download(final_wav_filename)

    print("\n🎉 TTS Task Completed Successfully!")
else:
    print("\n❌ An error occurred during TTS generation (See traceback above).")

In [ ]:
# @title 🎬 Video Generator & 📝 Subtitler!
# @markdown ✨ **Let's make some videos!** This cell scans your `/content/` folder for `.wav` audio files and images with matching names (like `Sarah.wav` and `Sarah.png`). It automatically pairs them up to create beautiful static videos and Whisper-subtitled videos. This will work seamlessly continuing on from the above if you uploded an image as your input!<br />
# @markdown 📂 *Tip:* `/content/` is the default spot for generated files, but you can also manually upload your own matching pairs using the "Files" menu on the left sidebar! <br />
# @markdown ⏱️ **A quick note on timing:** Processing times depend *heavily* on your audio length. Here is what to expect: <br />
# @markdown 🎬 **Standard Video (No Subtitles):** Takes about 1 minute to process for every 14 minutes of audio. <br />
# @markdown 💬 **Subtitled Video:** Takes about 1 minute to process for every 2.5 minutes of audio (or 1 min per 5 mins of audio with the "Faster" setting enabled). <br /> <br />
# @markdown *Note:* This step only needs an image and a .wav and optionally a .srt, if a .srt is not provided it will generate you one!<br />
# @markdown *Note 2:* If the Subtitles (.srt file) is wrong, typically the auto generated one is REALLY good, however it may fall over made up words, most notably, Names! Simply edit the .srt file, Delete the generated subtitled_video, and re-run this step it will use your existing .srt file and not generate a new one.
# @markdown <hr />
# @markdown ### ⚙️ Output & Subtitle Settings:
output_mode = "Both - With and without subtitles" # @param ["Video only (No Subtitles)", "Subtitled Video only", "Both - With and without subtitles"]
whisper_model_size = "large" # @param ["tiny", "base", "small", "medium", "large"]
# @markdown <hr />

# @markdown ## ⚡ Faster Generation:
# @markdown Tick this box to roughly halve the processing time for subtitled videos but at the cost of maybe very slightly misaligning sound to subtitles (No more than 0.17 seconds out!).
fast_subtitles = False # @param {type:"boolean"}

import os
import glob
import subprocess
import sys
import re
import zipfile

# --- 1. Install Required Libraries ---
try:
    from faster_whisper import WhisperModel
    import torch
except ImportError:
    print("📦 Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faster-whisper"], check=True)
    from faster_whisper import WhisperModel
    import torch

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tqdm"], check=True)
    from tqdm.auto import tqdm

# Import Colab file download utility
try:
    from google.colab import files
except ImportError:
    print("⚠️ google.colab not found (you are likely not running this in Google Colab). Downloads will be skipped.")
    files = None


# --- 2. Helper Functions ---
def get_duration(file_path):
    cmd = [
        "ffprobe", "-v", "error", "-show_entries",
        "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", file_path
    ]
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        return float(result.stdout.strip())
    except (subprocess.CalledProcessError, ValueError):
        return 0.0

def run_ffmpeg_with_progress(cmd, duration_sec, desc):
    process = subprocess.Popen(cmd, stderr=subprocess.PIPE, universal_newlines=True, encoding='utf-8')
    time_pattern = re.compile(r"time=(\d+):(\d+):(\d+\.\d+)")

    last_time = 0.0
    error_log = []
    bar_fmt = "{desc}: {percentage:3.0f}%|{bar}| {n:.1f}/{total:.1f}s [{elapsed}<{remaining}]"

    with tqdm(total=duration_sec, desc=desc, unit="s", bar_format=bar_fmt) as pbar:
        for line in process.stderr:
            error_log.append(line)
            if len(error_log) > 15:
                error_log.pop(0)

            match = time_pattern.search(line)
            if match:
                h, m, s = match.groups()
                current_time = int(h) * 3600 + int(m) * 60 + float(s)
                inc = current_time - last_time
                if inc > 0:
                    if last_time + inc > duration_sec:
                        pbar.update(duration_sec - last_time)
                    else:
                        pbar.update(inc)
                    last_time = current_time

        process.wait()
        if process.returncode == 0 and last_time < duration_sec:
            pbar.update(duration_sec - last_time)

    if process.returncode != 0:
        error_msg = "".join(error_log)
        raise RuntimeError(f"FFmpeg failed with error:\n{error_msg}")

def format_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    mills = int((seconds - int(seconds)) * 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{mills:03}"


# --- 3. Configuration & Model Loading ---
directory = "/content/"
image_extensions = ['.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff']

# Determine what needs to be generated based on the dropdown
generate_standard = output_mode in ["Video only (No Subtitles)", "Both - With and without subtitles"]
generate_subtitled = output_mode in ["Subtitled Video only", "Both - With and without subtitles"]

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

if device == "cpu":
    print("\n🚨 WARNING: No GPU detected! Whisper is running on CPU, which is VERY slow.")
    print("👉 To fix: Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU\n")

print(f"🔍 Scanning {directory} for matching audio and image files...")
wav_files = glob.glob(os.path.join(directory, "*.wav"))

# Track output groups per set: list of dicts with key elements
file_sets = []

if not wav_files:
    print("⚠️ No .wav files found in the directory.")
else:
    # Pre-scan to check if we actually need Whisper for any of the files
    need_whisper = False
    if generate_subtitled:
        for wav_path in wav_files:
            base_name = os.path.splitext(wav_path)[0]
            img_exists = any(os.path.exists(base_name + ext) for ext in image_extensions)

            if img_exists:
                vid2_path = base_name + "_subtitled_video.mp4"
                srt_path = base_name + ".srt"

                has_subtitled = os.path.exists(vid2_path) and os.path.getsize(vid2_path) > 0
                has_srt = os.path.exists(srt_path) and os.path.getsize(srt_path) > 0

                if not has_subtitled and not has_srt:
                    need_whisper = True
                    break

    model = None
    if need_whisper:
        print(f"⏳ Loading whisper '{whisper_model_size}' model onto {device.upper()}...")
        model = WhisperModel(whisper_model_size, device=device, compute_type=compute_type)
    elif generate_subtitled:
        print("⏭️ No new transcriptions needed (existing .srt files found for all items). Skipping Whisper load.")

    for wav_path in wav_files:
        base_name = os.path.splitext(wav_path)[0]
        base_filename = os.path.basename(base_name)

        img_path = None
        for ext in image_extensions:
            potential_img = base_name + ext
            if os.path.exists(potential_img):
                img_path = potential_img
                break

        if not img_path:
            continue

        vid1_path = base_name + "_video.mp4"
        vid2_path = base_name + "_subtitled_video.mp4"
        srt_path = base_name + ".srt"

        current_set_files = [wav_path, img_path]

        # Check if the requested videos already exist
        has_standard = os.path.exists(vid1_path) and os.path.getsize(vid1_path) > 0
        has_subtitled = os.path.exists(vid2_path) and os.path.getsize(vid2_path) > 0

        if (not generate_standard or has_standard) and (not generate_subtitled or has_subtitled):
            print(f"⏩ Skipping '{base_filename}' - Requested videos already exist.")
            if generate_standard and has_standard: current_set_files.append(vid1_path)
            if generate_subtitled and has_subtitled:
                current_set_files.append(vid2_path)
                if os.path.exists(srt_path): current_set_files.append(srt_path)
            file_sets.append({"name": base_filename, "files": current_set_files})
            continue

        print(f"\n🎬 Processing matched pair: '{base_filename}'")
        audio_duration = get_duration(wav_path)

        # --- 4. Generate Video 1 (Standard) ---
        if generate_standard and not has_standard:
            cmd1 = [
                "ffmpeg", "-y", "-loop", "1", "-framerate", "1", "-i", img_path, "-i", wav_path,
                "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
                "-c:v", "libx264", "-tune", "stillimage", "-preset", "ultrafast",
                "-vsync", "0",
                "-c:a", "aac", "-b:a", "192k",
                "-pix_fmt", "yuv420p", "-shortest", vid1_path
            ]
            try:
                run_ffmpeg_with_progress(cmd1, audio_duration, "    🎥 Creating base video")
                current_set_files.append(vid1_path)
            except RuntimeError as e:
                print(f"    ❌ {e}")
                continue

        # --- 5. Generate Video 2 (Subtitled) ---
        if generate_subtitled and not has_subtitled:
            # Check for an existing .srt file first
            if os.path.exists(srt_path) and os.path.getsize(srt_path) > 0:
                print(f"    📝 Found existing '{os.path.basename(srt_path)}'. Skipping Whisper transcription and using this file.")
            else:
                print("    📝 Transcribing audio with whisper...")
                segments, info = model.transcribe(wav_path, beam_size=5)

                with open(srt_path, "w", encoding="utf-8") as srt_file:
                    for i, segment in enumerate(segments, start=1):
                        start = format_timestamp(segment.start)
                        end = format_timestamp(segment.end)
                        text = segment.text.strip()
                        srt_file.write(f"{i}\n{start} --> {end}\n{text}\n\n")

            current_set_files.append(srt_path)
            safe_srt_path = srt_path.replace("\\", "/").replace(":", "\\:")
            subtitle_framerate = "3" if fast_subtitles else "6"

            cmd2 = [
                "ffmpeg", "-y", "-loop", "1", "-framerate", subtitle_framerate, "-i", img_path, "-i", wav_path,
                "-vf", f"scale=trunc(iw/2)*2:trunc(ih/2)*2,subtitles='{safe_srt_path}'",
                "-c:v", "libx264", "-preset", "ultrafast",
                "-vsync", "0",
                "-c:a", "aac", "-b:a", "192k",
                "-pix_fmt", "yuv420p", "-shortest", vid2_path
            ]

            try:
                run_ffmpeg_with_progress(cmd2, audio_duration, "    ✍️ Burning subtitles ")
                current_set_files.append(vid2_path)
            except RuntimeError as e:
                print(f"    ❌ {e}")
                continue

        # Save set files for zipping/downloading
        file_sets.append({"name": base_filename, "files": current_set_files})

# --- 6. Separate Zip & Download Logic ---
if files and file_sets:
    for item in file_sets:
        set_name = item["name"]
        set_files = list(set([f for f in item["files"] if os.path.exists(f)]))

        if set_files:
            zip_filename = f"{set_name}.zip"
            zip_path = os.path.join(directory, zip_filename)

            print(f"\n📦 Packaging {len(set_files)} files into {zip_filename} archive...")
            with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for file in set_files:
                    zipf.write(file, arcname=os.path.basename(file))

            print(f"📥 Triggering download for {zip_filename}...")
            files.download(zip_path)

print("\n🎉 All video processing tasks completed successfully!")